<a href="https://colab.research.google.com/github/Titantus/The-T0C-Predictive-Routing-Engine/blob/main/Two_Pendulums.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coupled-Pendulum Synchronization: Controlled Classical-Mechanics Test of the T'Z₀C Mismatch-Routing Hypothesis

This notebook models two damped, coupled pendulums to test the T'Z₀C mismatch-routing hypothesis within a classical-mechanics framework. It specifically evaluates the signed coupling-power term:

$$F_{\mathrm{sync}} = k_{\mathrm{eff}} \Delta\theta (\omega_2 - \omega_1)$$

where:

$$ \Delta\theta = \theta_2 - \theta_1 $$

Here, $F_{\mathrm{sync}}$ is treated as the **instantaneous signed power associated with mismatch-correcting coupling**. It represents the rate of power transfer due to the coupling mechanism within the system. Its sign indicates the direction of energy flow (into or out of the collective mechanical energy of the pendulums via coupling). Critically, for the purpose of a closed-system energy accounting, the absolute value of $F_{\mathrm{sync}}$ ($|F_{\mathrm{sync}}|$) is considered a power transfer that contributes to a 'Residue Heat Ledger' (also referred to as '0p Heat') alongside conventional damping losses.

## Scientific Hierarchy

This notebook tests the T'Z₀C mismatch-routing hypothesis in three progressively stronger stages, comparing the hypothesis-driven model against classical and non-thresholded controls:

1.  **Classical baseline** — constant coupling between damped pendulums ($k_{\mathrm{eff}} = k_{\mathrm{base}}$).
2.  **T'Z₀C mapping** — mismatch-dependent coupling, where the effective coupling strength is modulated by a threshold based on the framework primitive $p_c$. The value of $p_c$ is mapped to the `mismatch_threshold` variable in the simulation:
    $$p_c = \frac{5}{32\pi} \approx 0.04974\ \mathrm{rad}.$$
3.  **Nonlinear Control** — a continuous nonlinear coupling function used to determine whether observed behavior is specific to the thresholding mechanism or can be reproduced by ordinary continuous nonlinear coupling.

The value of $p_c$ is used as a **model-defined numerical mapping for the Vector/Wedge domain**. The notebook does not derive $p_c$, establish it as a universal saturation constant, or independently derive a physical 32-DOF capacity limit.

## Model and Energy Accounting

The model includes gravity, viscous damping, and mismatch-dependent coupling. It evaluates:

*   phase and velocity synchronization,
*   effective coupling ($k_{\mathrm{eff}}$),
*   $F_{\mathrm{sync}}$,
*   mechanical-energy evolution,
*   cumulative damping and coupling-power contributions, and
*   sensitivity to the mismatch threshold.

The implemented energy relation for the total mechanical energy of the pendulums ($E = E_1 + E_2$) is:

$$ \frac{dE}{dt} = -P_{\mathrm{damping}} - F_{\mathrm{sync}} $$

with the total damping power given by:

$$ P_{\mathrm{damping}} = b(\omega_1^2 + \omega_2^2). $$

It is critical to note that the definition of $E$ **explicitly excludes potential energy stored in the coupling mechanism itself**. Consequently, $F_{\mathrm{sync}}$ is interpreted as a **signed power-transfer term** directly affecting the pendulums' mechanical energy, distinct from an independent dissipative sink. For a comprehensive closed-system energy accounting, any energy transferred *from* the collective mechanical energy of the pendulums by $F_{\mathrm{sync}}$ is considered to be routed into a 'Residue Heat Ledger' (or '0p Heat'), represented by $|F_{\mathrm{sync}}|$. This approach ensures that the total energy lost from the pendulums' mechanical system ($-\frac{dE}{dt}$) is accounted for as either damping heat ($P_{\mathrm{damping}}$) or 'Residue Heat' ($|F_{\mathrm{sync}}|$) within a 'Triality of Modes' framework.

## Model Behavior

The simulation is used to determine whether thresholded coupling produces behavior that differs quantitatively and qualitatively from the classical and nonlinear controls. Relevant observables include:

*   synchronization rate and phase-lock time;
*   $|\Delta\theta|$ and $|\Delta\omega|$;
*   $k_{\mathrm{eff}}$;
*   instantaneous and cumulative $F_{\mathrm{sync}}$;
*   damping loss;
*   threshold-crossing frequency; and
*   sensitivity to $p_c$.

Observed synchronization, decay, or energy transfer alone does **not** establish the T'Z₀C interpretation, since these behaviors can occur in conventional coupled systems.

## Threshold Sensitivity Analysis for the T'Z₀C-inspired Model

The sensitivity sweep intentionally extends well beyond the framework value of $p_c$. This analysis focuses solely on the T'Z₀C-inspired/Thresholded model. Any saturation or diminishing-sensitivity region therefore describes the response of the implemented thresholded coupling law across a broad parameter range; it should **not** be interpreted as independent validation of $p_c$ itself. The primary goal is to characterize the behavior of the thresholded coupling. Its scope is limited to exploring the implemented coupling function's response to varying `mismatch_threshold` (representing $p_c$).

The key question is whether the $p_c$-based model produces a **quantitatively distinctive response** that cannot be reproduced by the classical or continuous-nonlinear controls.

## Relation to the Slip-Pivot Model

The pendulum system provides a controlled classical analogue for two qualitative features of the Aristotle Slip-Pivot model:

*   **Crest-and-decay:** strong early transients followed by reduced activity.
*   **Slip-and-lock:** reduced effective coupling during large mismatch followed by recovery as coherence develops.

The analogy is limited. The pendulum model does not implement the Slip-Pivot model's saturated hub, outer disk, or Tri-Valve Exhaust mechanisms, and it does not derive

$$ \sin\theta_0 \left(\frac{k_{\mathrm{proj}}}{4}\right) = \frac{2}{3\pi}. $$

## Interpretation and Status

The notebook is therefore a **controlled computational testbed**, not a proof of the framework. Its primary purpose is to determine whether a mismatch-thresholded coupling law produces a measurable signature beyond conventional coupled-pendulum dynamics.

**Status:** computationally testable hypothesis; T'Z₀C interpretation remains provisional pending a quantitative result that distinguishes the thresholded model from appropriate classical controls.

In [ ]:
# @title
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

plt.style.use("dark_background")

# Parameters
g, m, b = 9.81, 1.0, 0.1
L1, L2 = 1.0, 1.05
k_base = 0.5

# --- Defining and mapping the 'Grand Residual Saturation Threshold' ---
# As identified from the document search, the Grand Residual Saturation Threshold (p_c)
# is numerically defined as 5/(32*pi) radians.
# In this simulation, this value is explicitly mapped to and represents the
# '32-DOF capacity limit' for angular mismatch, affecting the effective coupling strength.
grand_residual_saturation_threshold_value = 5 / (32 * np.pi) # Value from T'Z0C documents

# The simulation's '32-DOF capacity limit' is now set by the 'Grand Residual Saturation Threshold'.
mismatch_threshold = grand_residual_saturation_threshold_value
# -------------------------------------------------------------------

def get_effective_coupling_classical(theta_diff, k_base, threshold=None):
    """Classical model: constant coupling."""
    return k_base

def get_effective_coupling_thresholded(theta_diff, k_base, threshold):
    """T'Z0C-inspired thresholded coupling."""
    abs_diff = np.abs(theta_diff)
    if abs_diff <= threshold:
        return k_base
    return k_base * max(0.1, threshold / abs_diff)

def get_effective_coupling_nonlinear_control(theta_diff, k_base, threshold):
    """Nonlinear Control model: continuous nonlinear coupling.
    The 'threshold' here acts as a characteristic scale for the nonlinearity.
    """
    abs_diff = np.abs(theta_diff)
    # A simple nonlinear form: k_eff decreases as mismatch increases
    # The division by 'threshold' normalizes the mismatch relative to this scale
    return k_base / (1 + (abs_diff / threshold)**2)

def pendulum_system(state, t, L1, L2, m, b, k_base, mismatch_threshold, coupling_model_func):
    θ1, ω1, θ2, ω2 = state
    Δθ = θ2 - θ1
    k_eff = coupling_model_func(Δθ, k_base, mismatch_threshold)

    α1 = (-m*g*L1*np.sin(θ1) - b*ω1 + k_eff*Δθ) / (m*L1**2)
    α2 = (-m*g*L2*np.sin(θ2) - b*ω2 - k_eff*Δθ) / (m*L2**2)
    return [ω1, α1, ω2, α2]

# Initial conditions & time
y0 = [np.pi/4, 0.0, np.pi/2, 0.0]
t = np.linspace(0, 50, 1000)

# --- Simulation for Classical Model ---
sol_classical = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, mismatch_threshold, get_effective_coupling_classical))
theta1_classical, omega1_classical, theta2_classical, omega2_classical = sol_classical.T

Delta_theta_classical = theta2_classical - theta1_classical
k_eff_classical = np.array([get_effective_coupling_classical(d, k_base, mismatch_threshold) for d in Delta_theta_classical])
F_sync_classical = k_eff_classical * Delta_theta_classical * (omega2_classical - omega1_classical)

# --- Simulation for Thresholded Model ---
sol_thresholded = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, mismatch_threshold, get_effective_coupling_thresholded))
theta1_thresholded, omega1_thresholded, theta2_thresholded, omega2_thresholded = sol_thresholded.T

Delta_theta_thresholded = theta2_thresholded - theta1_thresholded
k_eff_thresholded = np.array([get_effective_coupling_thresholded(d, k_base, mismatch_threshold) for d in Delta_theta_thresholded])
F_sync_thresholded = k_eff_thresholded * Delta_theta_thresholded * (omega2_thresholded - omega1_thresholded)

# --- Simulation for Nonlinear Control Model ---
sol_nonlinear = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, mismatch_threshold, get_effective_coupling_nonlinear_control))
theta1_nonlinear, omega1_nonlinear, theta2_nonlinear, omega2_nonlinear = sol_nonlinear.T

Delta_theta_nonlinear = theta2_nonlinear - theta1_nonlinear
k_eff_nonlinear = np.array([get_effective_coupling_nonlinear_control(d, k_base, mismatch_threshold) for d in Delta_theta_nonlinear])
F_sync_nonlinear = k_eff_nonlinear * Delta_theta_nonlinear * (omega2_nonlinear - omega1_nonlinear)

# Old simulation call (will be replaced in the next step)
# sol = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, mismatch_threshold))
# θ1, ω1, θ2, ω2 = sol.T

# # Effective coupling & F_sync
# Δθ = θ2 - θ1
# k_eff = np.array([get_effective_coupling(d, k_base, mismatch_threshold) for d in Δθ])
# F_sync = k_eff * Δθ * (ω2 - ω1)

# # Plots
# fig, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)

# axes[0].plot(t, θ1, label='Pendulum 1')
# axes[0].plot(t, θ2, label='Pendulum 2')
# axes[0].set(title='Pendulum Angles Over Time (with 32-DOF Capacity Limit set by Grand Residual Saturation Threshold)',
#             ylabel='Angle (rad)')
# axes[0].legend(); axes[0].grid(True)

# axes[1].plot(t, ω1, label='Pendulum 1')
# axes[1].plot(t, ω2, label='Pendulum 2')
# axes[1].set(title='Pendulum Angular Velocities Over Time',
#             ylabel='Angular Velocity (rad/s)')
# axes[1].legend(); axes[1].grid(True)

# axes[2].plot(t, k_eff, label='k_coupling_effective')
# axes[2].axhline(k_base, color='r', ls='--', label='Base Coupling Strength')
# axes[2].set(title='Effective Coupling Strength Over Time (Impact of Grand Residual Saturation Threshold)',
#             ylabel='k_coupling_effective')
# axes[2].legend(); axes[2].grid(True)

# axes[3].plot(t, F_sync, label=r'Proposed $F_{sync}$ (Mismatch Neutralization Power)')
# axes[3].set(title=r'Proposed $F_{sync}$ (Instantaneous Power for Mismatch Neutralization)',
#             xlabel='Time (s)', ylabel=r'$F_{sync}$ (W)')
# axes[3].legend(); axes[3].grid(True)

# plt.tight_layout()
# plt.show()

# # Define a range of mismatch thresholds to test (from very low to high)
# # The default 'mismatch_threshold' is set to the 'grand_residual_saturation_threshold_value'.
# # This sensitivity analysis explores the effect of varying this '32-DOF capacity limit'.
# mismatch_threshold_values = np.linspace(0.05 * np.pi, 0.5 * np.pi, 20) # Range from approx. 9 to 90 degrees
# cumulative_F_sync_losses = []

# print("Running sensitivity analysis...")
# for threshold in mismatch_threshold_values:
#     # Rerun the simulation for each threshold
#     sol_sens = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, threshold))

#     # Extract results
#     theta1_sens, omega1_sens, theta2_sens, omega2_sens = sol_sens.T

#     # Recalculate effective coupling and F_sync
#     delta_theta_sens = theta2_sens - theta1_sens
#     k_eff_sens = np.array([get_effective_coupling(d, k_base, threshold) for d in delta_theta_sens])
#     F_sync_sens = k_eff_sens * delta_theta_sens * (omega2_sens - omega1_sens)

#     # Calculate cumulative E_sync_cum for this run
#     E_sync_cum_sens = np.cumsum(F_sync_sens) * (t[1] - t[0])
#     cumulative_F_sync_losses.append(E_sync_cum_sens[-1])

# print("Analysis complete. Plotting results...")

# # Plot the sensitivity analysis results
# plt.figure(figsize=(10, 6))
# plt.plot(np.degrees(mismatch_threshold_values), cumulative_F_sync_losses, 'o-', color='skyblue')
# plt.xlabel('32-DOF Capacity Limit (degrees) - representing Grand Residual Saturation Threshold')
# plt.ylabel('Cumulative $F_{sync}$ Energy (J)')
# plt.title('Sensitivity of Cumulative $F_{sync}$ to 32-DOF Capacity Limit (Grand Residual Saturation Threshold)')
# plt.grid(True)
# plt.tight_layout()
# plt.show()

In [ ]:
# @title
PE1 = m * g * L1 * (1 - np.cos(theta1_thresholded))
KE1 = 0.5 * m * L1**2 * omega1_thresholded**2
PE2 = m * g * L2 * (1 - np.cos(theta2_thresholded))
KE2 = 0.5 * m * L2**2 * omega2_thresholded**2

TE1 = PE1 + KE1
TE2 = PE2 + KE2
Total_System_Energy = TE1 + TE2

# --- Energy components plot ---
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(t, PE1, label='PE1')
axes[0].plot(t, KE1, label='KE1')
axes[0].plot(t, TE1, '--', label='TE1')
axes[0].set(title='Pendulum 1 Energy Components (Thresholded Model)', ylabel='Energy (J)')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, PE2, label='PE2')
axes[1].plot(t, KE2, label='KE2')
axes[1].plot(t, TE2, '--', label='TE2')
axes[1].set(title='Pendulum 2 Energy Components (Thresholded Model)', ylabel='Energy (J)')
axes[1].legend(); axes[1].grid(True)

axes[2].plot(t, Total_System_Energy, color='purple', label='Total System Energy (Thresholded Model)')
axes[2].set(title='Total System Energy Over Time (Thresholded Model)', xlabel='Time (s)', ylabel='Energy (J)')
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.show()

# --- Damping power & energy-conservation check ---
# Using data from the Thresholded Model for consistency
P_damping_total_thresholded = b * (omega1_thresholded**2 + omega2_thresholded**2)
dTE_dt = np.gradient(Total_System_Energy, t)
# The energy equation: dE/dt = -P_damping - F_sync. F_sync is treated as a power transfer affecting mechanical energy.
expected_dTE_dt = -(P_damping_total_thresholded + F_sync_thresholded)

plt.figure(figsize=(12, 6))
plt.plot(t, dTE_dt, color='cyan', label='Actual d(Total_System_Energy)/dt')
plt.plot(t, expected_dTE_dt, 'r--', label='Expected dE/dt (damping + F_sync)')
plt.title('Comparison of Actual vs. Expected Rate of Change of Total System Energy (Thresholded Model)')
plt.xlabel('Time (s)')
plt.ylabel('Rate of Energy Change (W)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# @title
P_damping = b * (omega1_thresholded**2 + omega2_thresholded**2)  # Damping power from thresholded model (always >= 0)
P_sync_residue = np.abs(F_sync_thresholded)             # Absolute value of F_sync, routed to Residue Heat Ledger (0p Heat)

# Cumulative energy removed by each mechanism (from thresholded model)
E_damping_cum = np.cumsum(P_damping) * (t[1] - t[0])
E_sync_residue_cum = np.cumsum(P_sync_residue) * (t[1] - t[0])
E_residue_total = E_damping_cum + E_sync_residue_cum # Total energy routed to heat ledgers

# Energy that should have been lost according to the drop in Total_System_Energy (from thresholded model)
E_from_TE = Total_System_Energy[0] - Total_System_Energy

# Energy conservation check: Closed-System Sum should ideally be close to zero
Closed_System_Sum = E_from_TE - E_residue_total

# --- Plots --- #
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

# 1. Instantaneous powers
axes[0].plot(t, P_damping, label=r'$P_{\mathrm{damping}}$', color='C0')
axes[0].plot(t, P_sync_residue, label=r'$P_{\mathrm{sync, residue}}$ ($|F_{\mathrm{sync}}|$, 0p Heat)', alpha=0.8, color='C1')
axes[0].axhline(0, color='gray', lw=0.8)
axes[0].set(title='Instantaneous Energy Routing to Heat Ledgers (Thresholded Model)',
            ylabel='Power (W)')
axes[0].legend(); axes[0].grid(True)

# 2. Cumulative energy removed
axes[1].plot(t, E_damping_cum, label='Cumulative Damping (Heat Loss)', color='C0')
axes[1].plot(t, E_sync_residue_cum, label=r'Cumulative $P_{\mathrm{sync, residue}}$ (0p Heat)', color='C1')
axes[1].plot(t, E_residue_total, 'w--', label='Total Routed to Heat Ledgers')
axes[1].plot(t, E_from_TE, 'r:', label=r'Actual Drop in Mechanical Energy ($-\Delta E_{\mathrm{mech}}$)')
axes[1].plot(t, Closed_System_Sum, 'g-.', label=r'Closed-System Sum ($-\Delta E_{\mathrm{mech}} - \sum E_{\mathrm{heat}}$)')
axes[1].set(title='Cumulative Energy Routing (Thresholded Model)',
            ylabel='Energy (J)')
axes[1].legend(); axes[1].grid(True)

# 3. Relative contribution of each channel
total_routed_loss = E_residue_total[-1]
if total_routed_loss > 1e-9: # Avoid division by zero for very small losses
    frac_damp = E_damping_cum[-1] / total_routed_loss * 100
    frac_sync_residue = E_sync_residue_cum[-1] / total_routed_loss * 100
else:
    frac_damp = 0
    frac_sync_residue = 0

axes[2].bar(['Damping (Heat Loss)', r'$P_{\mathrm{sync, residue}}$ (0p Heat)'],
            [frac_damp, frac_sync_residue],
            color=['C0', 'C1'])
axes[2].set(title=f'Relative Contribution to Total Routed Heat ({total_routed_loss:.2f} J)',
            ylabel='Percentage (%)')
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.show()

print(f"Initial Total Mechanical Energy: {Total_System_Energy[0]:.3f} J")
print(f"Final Total Mechanical Energy  : {Total_System_Energy[-1]:.3f} J")
print(f"Total drop in Mechanical Energy: {E_from_TE[-1]:.3f} J")
print("--------------------------------------------------")
print(f"Cumulative Damping (Heat Loss) : {E_damping_cum[-1]:.3f} J  ({frac_damp:.1f}%) -- due to viscous friction")
print(f"Cumulative P_sync, residue (0p Heat) : {E_sync_residue_cum[-1]:.3f} J  ({frac_sync_residue:.1f}%) -- energy routed from coupling to 0p Heat")
print(f"Total Routed to Heat Ledgers   : {E_residue_total[-1]:.3f} J")
print("--------------------------------------------------")
print(f"Closed-System Sum (should be ~0): {Closed_System_Sum[-1]:.3f} J")

In [ ]:
# @title
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

plt.style.use("dark_background")

# Parameters
g, m, b = 9.81, 1.0, 0.1
L1, L2 = 1.0, 1.05
k_base = 0.5

# --- Defining and mapping the 'Grand Residual Saturation Threshold' ---
# As identified from the document search, the Grand Residual Saturation Threshold (p_c)
# is numerically defined as 5/(32*pi) radians.
# In this simulation, this value is explicitly mapped to and represents the
# '32-DOF capacity limit' for angular mismatch, affecting the effective coupling strength.
grand_residual_saturation_threshold_value = 5 / (32 * np.pi) # Value from T'Z0C documents

# The simulation's '32-DOF capacity limit' is now set by the 'Grand Residual Saturation Threshold'.
mismatch_threshold = grand_residual_saturation_threshold_value
# -------------------------------------------------------------------

def get_effective_coupling_classical(theta_diff, k_base, threshold=None):
    """Classical model: constant coupling."""
    return k_base

def get_effective_coupling_thresholded(theta_diff, k_base, threshold):
    """T'Z0C-inspired thresholded coupling."""
    abs_diff = np.abs(theta_diff)
    if abs_diff <= threshold:
        return k_base
    return k_base * max(0.1, threshold / abs_diff)

def get_effective_coupling_nonlinear_control(theta_diff, k_base, threshold):
    """Nonlinear Control model: continuous nonlinear coupling.
    The 'threshold' here acts as a characteristic scale for the nonlinearity.
    """
    abs_diff = np.abs(theta_diff)
    # A simple nonlinear form: k_eff decreases as mismatch increases
    # The division by 'threshold' normalizes the mismatch relative to this scale
    return k_base / (1 + (abs_diff / threshold)**2)

def pendulum_system(state, t, L1, L2, m, b, k_base, mismatch_threshold, coupling_model_func):
    θ1, ω1, θ2, ω2 = state
    Δθ = θ2 - θ1
    k_eff = coupling_model_func(Δθ, k_base, mismatch_threshold)

    α1 = (-m*g*L1*np.sin(θ1) - b*ω1 + k_eff*Δθ) / (m*L1**2)
    α2 = (-m*g*L2*np.sin(θ2) - b*ω2 - k_eff*Δθ) / (m*L2**2)
    return [ω1, α1, ω2, α2]

# Initial conditions & time
y0 = [np.pi/4, 0.0, np.pi/2, 0.0]
t = np.linspace(0, 50, 1000)

# --- Simulation for Classical Model ---
sol_classical = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, mismatch_threshold, get_effective_coupling_classical))
theta1_classical, omega1_classical, theta2_classical, omega2_classical = sol_classical.T

Delta_theta_classical = theta2_classical - theta1_classical
k_eff_classical = np.array([get_effective_coupling_classical(d, k_base, mismatch_threshold) for d in Delta_theta_classical])
F_sync_classical = k_eff_classical * Delta_theta_classical * (omega2_classical - omega1_classical)

# --- Simulation for Thresholded Model ---
sol_thresholded = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, mismatch_threshold, get_effective_coupling_thresholded))
theta1_thresholded, omega1_thresholded, theta2_thresholded, omega2_thresholded = sol_thresholded.T

Delta_theta_thresholded = theta2_thresholded - theta1_thresholded
k_eff_thresholded = np.array([get_effective_coupling_thresholded(d, k_base, mismatch_threshold) for d in Delta_theta_thresholded])
F_sync_thresholded = k_eff_thresholded * Delta_theta_thresholded * (omega2_thresholded - omega1_thresholded)

# --- Simulation for Nonlinear Control Model ---
sol_nonlinear = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, mismatch_threshold, get_effective_coupling_nonlinear_control))
theta1_nonlinear, omega1_nonlinear, theta2_nonlinear, omega2_nonlinear = sol_nonlinear.T

Delta_theta_nonlinear = theta2_nonlinear - theta1_nonlinear
k_eff_nonlinear = np.array([get_effective_coupling_nonlinear_control(d, k_base, mismatch_threshold) for d in Delta_theta_nonlinear])
F_sync_nonlinear = k_eff_nonlinear * Delta_theta_nonlinear * (omega2_nonlinear - omega1_nonlinear)


# --- Generate Comparative Plots ---
fig, axes = plt.subplots(4, 1, figsize=(12, 20), sharex=True)

# Plot 1: Pendulum Angles
axes[0].plot(t, theta1_classical, label='P1 Classical', color='red', linestyle='-')
axes[0].plot(t, theta2_classical, label='P2 Classical', color='red', linestyle='--')
axes[0].plot(t, theta1_thresholded, label='P1 T_Z0C', color='green', linestyle='-')
axes[0].plot(t, theta2_thresholded, label='P2 T_Z0C', color='green', linestyle='--')
axes[0].plot(t, theta1_nonlinear, label='P1 Nonlinear', color='blue', linestyle='-')
axes[0].plot(t, theta2_nonlinear, label='P2 Nonlinear', color='blue', linestyle='--')
axes[0].set_ylabel('Angle (rad)')
axes[0].set_title('Comparative Pendulum Angles Over Time for Different Coupling Models')
axes[0].legend(loc='upper right', ncol=3)
axes[0].grid(True)

# Plot 2: Pendulum Angular Velocities
axes[1].plot(t, omega1_classical, label='P1 Classical', color='red', linestyle='-')
axes[1].plot(t, omega2_classical, label='P2 Classical', color='red', linestyle='--')
axes[1].plot(t, omega1_thresholded, label='P1 T_Z0C', color='green', linestyle='-')
axes[1].plot(t, omega2_thresholded, label='P2 T_Z0C', color='green', linestyle='--')
axes[1].plot(t, omega1_nonlinear, label='P1 Nonlinear', color='blue', linestyle='-')
axes[1].plot(t, omega2_nonlinear, label='P2 Nonlinear', color='blue', linestyle='--')
axes[1].set_ylabel('Angular Velocity (rad/s)')
axes[1].set_title('Comparative Angular Velocities Over Time for Different Coupling Models')
axes[1].legend(loc='upper right', ncol=3)
axes[1].grid(True)

# Plot 3: Effective Coupling Strength
axes[2].plot(t, k_eff_classical, label='Classical', color='red')
axes[2].plot(t, k_eff_thresholded, label="T'Z0C-inspired/Thresholded", color='green')
axes[2].plot(t, k_eff_nonlinear, label='Nonlinear Control', color='blue')
axes[2].axhline(k_base, color='gray', linestyle=':', label='Base Coupling Strength')
axes[2].set_ylabel('Effective Coupling')
axes[2].set_title('Comparative Effective Coupling Strength Over Time')
axes[2].legend(loc='upper right')
axes[2].grid(True)

# Plot 4: F_sync
axes[3].plot(t, F_sync_classical, label='Classical', color='red')
axes[3].plot(t, F_sync_thresholded, label="T'Z0C-inspired/Thresholded", color='green')
axes[3].plot(t, F_sync_nonlinear, label='Nonlinear Control', color='blue')
axes[3].axhline(0, color='gray', linestyle=':')
axes[3].set_xlabel('Time (s)')
axes[3].set_ylabel(r'$F_{sync}$ (W)')
axes[3].set_title(r'Comparative $F_{sync}$ (Instantaneous Power) Over Time')
axes[3].legend(loc='upper right')
axes[3].grid(True)

plt.tight_layout()
plt.show()


# --- Sensitivity Analysis for T'Z₀C-inspired/Thresholded Model ---
# This analysis specifically focuses on the T'Z₀C-inspired/Thresholded coupling model.
# It explores how the implemented coupling function responds to variations in the
# `mismatch_threshold` parameter, which represents the T'Z₀C framework's `p_c` value.
# Note that the true `p_c` is approximately 0.0497 radians (2.85 degrees).
# The current starting point of this sensitivity sweep (0.05 * np.pi) is approximately 9 degrees,
# which is significantly higher than the true `p_c`.
mismatch_threshold_values = np.linspace(0.05 * np.pi, 0.5 * np.pi, 20) # Range from approx. 9 to 90 degrees
cumulative_F_sync_losses = []

print("Running sensitivity analysis...")
for threshold in mismatch_threshold_values:
    # Rerun the simulation for each threshold
    sol_sens = odeint(pendulum_system, y0, t, args=(L1, L2, m, b, k_base, threshold, get_effective_coupling_thresholded)) # Use thresholded model for sensitivity analysis

    # Extract results
    theta1_sens, omega1_sens, theta2_sens, omega2_sens = sol_sens.T

    # Recalculate effective coupling and F_sync
    delta_theta_sens = theta2_sens - theta1_sens
    k_eff_sens = np.array([get_effective_coupling_thresholded(d, k_base, threshold) for d in delta_theta_sens])
    F_sync_sens = k_eff_sens * delta_theta_sens * (omega2_sens - omega1_sens)

    # Calculate cumulative E_sync_cum for this run
    E_sync_cum_sens = np.cumsum(F_sync_sens) * (t[1] - t[0])
    cumulative_F_sync_losses.append(E_sync_cum_sens[-1])

print("Analysis complete. Plotting results...")

# Plot the sensitivity analysis results
plt.figure(figsize=(10, 6))
plt.plot(np.degrees(mismatch_threshold_values), cumulative_F_sync_losses, 'o-', color='skyblue')
plt.xlabel('32-DOF Capacity Limit (degrees) - representing Grand Residual Saturation Threshold')
plt.ylabel('Cumulative $F_{sync}$ Energy (J)')
plt.title('Sensitivity of Cumulative $F_{sync}$ to 32-DOF Capacity Limit (Grand Residual Saturation Threshold) - Thresholded Model')
plt.grid(True)
plt.tight_layout()
plt.show()